In [0]:
WITH params AS (
  SELECT
    date_trunc('MONTH', current_date()) AS this_month_start,
    date_trunc('YEAR',  current_date()) AS year_start
),
bounds AS (
  SELECT
    (SELECT year_start       FROM params) AS cy_start_incl,
    (SELECT this_month_start FROM params) AS cy_end_excl,

    add_months((SELECT year_start       FROM params), -12) AS py_start_incl,
    add_months((SELECT this_month_start FROM params), -12) AS py_end_excl
),

/* -----------------------------
   Visitor-quarter classification (Option A)
   - One row per (period, quarter_start, visitor_id)
   - is_new_visitor_q = 1 if visitor had ANY session with is_new_visitor=true in that quarter
   ----------------------------- */
visitor_q AS (
  SELECT
    CASE
      WHEN to_date(sp.date) >= (SELECT cy_start_incl FROM bounds)
       AND to_date(sp.date) <  (SELECT cy_end_excl  FROM bounds) THEN 'CY'
      WHEN to_date(sp.date) >= (SELECT py_start_incl FROM bounds)
       AND to_date(sp.date) <  (SELECT py_end_excl  FROM bounds) THEN 'PY'
      ELSE NULL
    END AS period,
    date_trunc('QUARTER', to_date(sp.date)) AS quarter_start,
    sp.visitor_id,
    max(CASE WHEN coalesce(sp.is_new_visitor, false) THEN 1 ELSE 0 END) AS is_new_visitor_q
  FROM production.MARKETPLACE_REPORTS.AGG_SESSION_PERFORMANCE sp
  WHERE
    (
      to_date(sp.date) >= (SELECT cy_start_incl FROM bounds)
      AND to_date(sp.date) <  (SELECT cy_end_excl  FROM bounds)
    )
    OR
    (
      to_date(sp.date) >= (SELECT py_start_incl FROM bounds)
      AND to_date(sp.date) <  (SELECT py_end_excl  FROM bounds)
    )
  GROUP BY 1,2,3
),
visitor_q_agg AS (
  SELECT
    period,
    quarter_start,
    COUNT(*) AS visitors,
    SUM(is_new_visitor_q) AS new_visitors,
    COUNT(*) - SUM(is_new_visitor_q) AS returning_visitors
  FROM visitor_q
  WHERE period IS NOT NULL
  GROUP BY 1,2
),

/* -----------------------------
   Sessions data (session-level metrics only)
   ----------------------------- */
sessions_q AS (
  SELECT
    CASE
      WHEN to_date(sp.date) >= (SELECT cy_start_incl FROM bounds)
       AND to_date(sp.date) <  (SELECT cy_end_excl  FROM bounds) THEN 'CY'
      WHEN to_date(sp.date) >= (SELECT py_start_incl FROM bounds)
       AND to_date(sp.date) <  (SELECT py_end_excl  FROM bounds) THEN 'PY'
      ELSE NULL
    END AS period,
    date_trunc('QUARTER', to_date(sp.date)) AS quarter_start,

    COUNT(DISTINCT sp.session_id) AS sessions,
    COUNT(DISTINCT IF(sp.is_bounce, sp.session_id, NULL)) AS bounced_sessions,
    COUNT(DISTINCT IF(sp.adp_views > 0, sp.visitor_id, NULL)) AS quoted_visitors
  FROM production.MARKETPLACE_REPORTS.AGG_SESSION_PERFORMANCE sp
  WHERE
    (
      to_date(sp.date) >= (SELECT cy_start_incl FROM bounds)
      AND to_date(sp.date) <  (SELECT cy_end_excl  FROM bounds)
    )
    OR
    (
      to_date(sp.date) >= (SELECT py_start_incl FROM bounds)
      AND to_date(sp.date) <  (SELECT py_end_excl  FROM bounds)
    )
  GROUP BY 1,2
),
sessions_q_final AS (
  SELECT
    s.period,
    s.quarter_start,

    s.sessions,
    v.visitors,
    v.new_visitors,
    v.returning_visitors,

    (s.bounced_sessions / NULLIF(s.sessions, 0)) AS bounce_rate,
    (s.quoted_visitors  / NULLIF(v.visitors, 0)) AS quoter_rate,
    (v.new_visitors     / NULLIF(v.visitors, 0)) AS new_visitor_share
  FROM sessions_q s
  LEFT JOIN visitor_q_agg v
    ON v.period = s.period
   AND v.quarter_start = s.quarter_start
  WHERE s.period IS NOT NULL
),

/* -----------------------------
   Bookings + cart join (single pass)
   Commission rate:
   - fb.commission_rate appears to be stored in PERCENT POINTS (e.g. 10, 20)
   - We compute a GMV-weighted average in the SAME UNIT (percent points)
   ----------------------------- */
bookings_q AS (
  SELECT
    CASE
      WHEN to_date(fb.date_of_checkout) >= (SELECT cy_start_incl FROM bounds)
       AND to_date(fb.date_of_checkout) <  (SELECT cy_end_excl  FROM bounds) THEN 'CY'
      WHEN to_date(fb.date_of_checkout) >= (SELECT py_start_incl FROM bounds)
       AND to_date(fb.date_of_checkout) <  (SELECT py_end_excl  FROM bounds) THEN 'PY'
      ELSE NULL
    END AS period,
    date_trunc('QUARTER', to_date(fb.date_of_checkout)) AS quarter_start,

    COUNT(DISTINCT CASE WHEN coalesce(fb.status_id, -1) = 1 THEN fb.booking_id END)        AS bookings,
    COUNT(DISTINCT CASE WHEN coalesce(fb.status_id, -1) = 1 THEN fb.shopping_cart_id END) AS transactions,

    COUNT(DISTINCT CASE
      WHEN coalesce(fb.status_id, -1) = 1
       AND coalesce(sc.coupon_id, 0) > 0
      THEN fb.booking_id
    END) AS bookings_with_coupon,

    SUM(CASE WHEN coalesce(fb.status_id, -1) = 1 THEN coalesce(fb.gmv, 0) ELSE 0 END)               AS gmv_eur,
    SUM(CASE WHEN coalesce(fb.status_id, -1) = 1 THEN coalesce(fb.nr, 0) ELSE 0 END)                AS nr_eur,
    SUM(CASE WHEN coalesce(fb.status_id, -1) = 1 THEN coalesce(fb.nr_supplier_share, 0) ELSE 0 END) AS nr_bt_eur,
    SUM(CASE WHEN coalesce(fb.status_id, -1) = 1 THEN coalesce(fb.net_tax_eur, 0) ELSE 0 END)       AS net_tax_eur,

    SUM(CASE WHEN coalesce(fb.status_id, -1) = 1 THEN coalesce(fb.price_net_rate_eur, 0) ELSE 0 END) AS price_net_rate_eur_sum,

    /* GMV-weighted commission rate in PERCENT POINTS */
    SUM(CASE WHEN coalesce(fb.status_id, -1) = 1
             THEN coalesce(fb.commission_rate, 0) * coalesce(fb.gmv, 0)
             ELSE 0 END) AS comm_rate_pp_x_gmv_sum,
    SUM(CASE WHEN coalesce(fb.status_id, -1) = 1 THEN coalesce(fb.gmv, 0) ELSE 0 END) AS comm_rate_weight_gmv_sum

  FROM production.dwh.fact_booking_v2 fb
  LEFT JOIN production.dwh.fact_shopping_cart sc
    ON sc.shopping_cart_id = fb.shopping_cart_id
  WHERE fb.date_of_checkout IS NOT NULL
    AND (
      (
        to_date(fb.date_of_checkout) >= (SELECT cy_start_incl FROM bounds)
        AND to_date(fb.date_of_checkout) <  (SELECT cy_end_excl  FROM bounds)
      )
      OR
      (
        to_date(fb.date_of_checkout) >= (SELECT py_start_incl FROM bounds)
        AND to_date(fb.date_of_checkout) <  (SELECT py_end_excl  FROM bounds)
      )
    )
  GROUP BY 1,2
),
bookings_q_final AS (
  SELECT *
  FROM bookings_q
  WHERE period IS NOT NULL
),

/* -----------------------------
   Combine sessions + bookings per (period, quarter)
   ----------------------------- */
q_all AS (
  SELECT
    coalesce(s.period, b.period) AS period,
    coalesce(s.quarter_start, b.quarter_start) AS quarter_start,

    s.sessions,
    s.visitors,
    s.new_visitors,
    s.returning_visitors,
    s.new_visitor_share,
    s.bounce_rate,
    s.quoter_rate,

    b.bookings,
    b.transactions,
    b.bookings_with_coupon,

    CASE WHEN b.bookings = 0 THEN NULL ELSE b.bookings_with_coupon / b.bookings END AS bookings_with_coupon_share,

    b.gmv_eur,
    b.nr_eur,
    b.nr_bt_eur,
    b.net_tax_eur,

    CASE WHEN b.gmv_eur = 0 THEN NULL ELSE b.nr_eur    / b.gmv_eur END AS take_rate_nr,
    CASE WHEN b.gmv_eur = 0 THEN NULL ELSE b.nr_bt_eur / b.gmv_eur END AS take_rate_nr_bt,

    CASE WHEN b.transactions = 0 THEN NULL ELSE b.gmv_eur / b.transactions END AS aov_eur,
    CASE WHEN b.bookings     = 0 THEN NULL ELSE b.gmv_eur / b.bookings     END AS avg_price_eur,

    /* Commission rate in percent points (e.g., 10.00 means 10%) */
    CASE WHEN b.comm_rate_weight_gmv_sum = 0 THEN NULL
         ELSE b.comm_rate_pp_x_gmv_sum / b.comm_rate_weight_gmv_sum
    END AS commission_rate,

    CASE WHEN s.sessions = 0 THEN NULL ELSE b.bookings / s.sessions END AS conversion_rate

  FROM sessions_q_final s
  FULL OUTER JOIN bookings_q_final b
    ON b.period = s.period
   AND b.quarter_start = s.quarter_start
),

cy AS (SELECT * FROM q_all WHERE period = 'CY'),
py AS (SELECT * FROM q_all WHERE period = 'PY'),

/* -----------------------------
   Final wide-by-quarter rows (one row per CY quarter)
   ----------------------------- */
final AS (
  SELECT
    cy.quarter_start,
    concat('Q', quarter(cy.quarter_start), ' - ', year(cy.quarter_start)) AS quarter_label,

    -- CY
    cy.sessions,
    cy.visitors,
    cy.new_visitors,
    cy.returning_visitors,
    cy.new_visitor_share,
    cy.bounce_rate,
    cy.quoter_rate,

    cy.bookings,
    cy.transactions,
    cy.bookings_with_coupon,
    cy.bookings_with_coupon_share,

    cy.gmv_eur,
    cy.nr_eur,
    cy.nr_bt_eur,
    cy.net_tax_eur,
    cy.take_rate_nr,
    cy.take_rate_nr_bt,
    cy.aov_eur,
    cy.avg_price_eur,
    cy.commission_rate,
    cy.conversion_rate,

    -- PY
    py.sessions        AS py_sessions,
    py.visitors        AS py_visitors,
    py.new_visitors    AS py_new_visitors,
    py.returning_visitors AS py_returning_visitors,
    py.new_visitor_share AS py_new_visitor_share,
    py.bounce_rate     AS py_bounce_rate,
    py.quoter_rate     AS py_quoter_rate,

    py.bookings        AS py_bookings,
    py.transactions    AS py_transactions,
    py.bookings_with_coupon AS py_bookings_with_coupon,
    py.bookings_with_coupon_share AS py_bookings_with_coupon_share,

    py.gmv_eur         AS py_gmv_eur,
    py.nr_eur          AS py_nr_eur,
    py.nr_bt_eur       AS py_nr_bt_eur,
    py.net_tax_eur     AS py_net_tax_eur,
    py.take_rate_nr    AS py_take_rate_nr,
    py.take_rate_nr_bt AS py_take_rate_nr_bt,
    py.aov_eur         AS py_aov_eur,
    py.avg_price_eur   AS py_avg_price_eur,
    py.commission_rate AS py_commission_rate,
    py.conversion_rate AS py_conversion_rate,

    -- YoY abs (volumes)
    (cy.sessions     - py.sessions)     AS yoy_sessions_abs,
    (cy.visitors     - py.visitors)     AS yoy_visitors_abs,
    (cy.new_visitors - py.new_visitors) AS yoy_new_visitors_abs,
    (cy.returning_visitors - py.returning_visitors) AS yoy_returning_visitors_abs,

    (cy.bookings     - py.bookings)     AS yoy_bookings_abs,
    (cy.transactions - py.transactions) AS yoy_transactions_abs,
    (cy.bookings_with_coupon - py.bookings_with_coupon) AS yoy_bookings_with_coupon_abs,

    (cy.gmv_eur      - py.gmv_eur)      AS yoy_gmv_eur_abs,
    (cy.nr_eur       - py.nr_eur)       AS yoy_nr_eur_abs,
    (cy.nr_bt_eur    - py.nr_bt_eur)    AS yoy_nr_bt_eur_abs,
    (cy.net_tax_eur  - py.net_tax_eur)  AS yoy_net_tax_eur_abs,

    -- YoY abs (rates/prices)
    (cy.aov_eur         - py.aov_eur)          AS yoy_aov_eur_abs,
    (cy.avg_price_eur   - py.avg_price_eur)    AS yoy_avg_price_eur_abs,
    (cy.commission_rate - py.commission_rate)  AS yoy_commission_rate_abs,      -- percent points
    (cy.conversion_rate - py.conversion_rate)  AS yoy_conversion_rate_abs,      -- fraction
    (cy.new_visitor_share - py.new_visitor_share) AS yoy_new_visitor_share_abs, -- fraction
    (cy.bookings_with_coupon_share - py.bookings_with_coupon_share) AS yoy_bookings_with_coupon_share_abs, -- fraction

    -- YoY % (volumes)
    CASE WHEN py.sessions     = 0 THEN NULL ELSE (cy.sessions     / py.sessions)     - 1 END AS yoy_sessions_pct,
    CASE WHEN py.visitors     = 0 THEN NULL ELSE (cy.visitors     / py.visitors)     - 1 END AS yoy_visitors_pct,
    CASE WHEN py.new_visitors = 0 THEN NULL ELSE (cy.new_visitors / py.new_visitors) - 1 END AS yoy_new_visitors_pct,
    CASE WHEN py.returning_visitors = 0 THEN NULL ELSE (cy.returning_visitors / py.returning_visitors) - 1 END AS yoy_returning_visitors_pct,

    CASE WHEN py.bookings     = 0 THEN NULL ELSE (cy.bookings     / py.bookings)     - 1 END AS yoy_bookings_pct,
    CASE WHEN py.transactions = 0 THEN NULL ELSE (cy.transactions / py.transactions) - 1 END AS yoy_transactions_pct,
    CASE WHEN py.bookings_with_coupon = 0 THEN NULL ELSE (cy.bookings_with_coupon / py.bookings_with_coupon) - 1 END AS yoy_bookings_with_coupon_pct,

    CASE WHEN py.gmv_eur      = 0 THEN NULL ELSE (cy.gmv_eur      / py.gmv_eur)      - 1 END AS yoy_gmv_eur_pct,
    CASE WHEN py.nr_eur       = 0 THEN NULL ELSE (cy.nr_eur       / py.nr_eur)       - 1 END AS yoy_nr_eur_pct,
    CASE WHEN py.nr_bt_eur    = 0 THEN NULL ELSE (cy.nr_bt_eur    / py.nr_bt_eur)    - 1 END AS yoy_nr_bt_eur_pct,
    CASE WHEN py.net_tax_eur  = 0 THEN NULL ELSE (cy.net_tax_eur  / py.net_tax_eur)  - 1 END AS yoy_net_tax_eur_pct,

    -- YoY % (rates/prices)
    CASE WHEN py.aov_eur = 0 THEN NULL ELSE (cy.aov_eur / py.aov_eur) - 1 END AS yoy_aov_pct,
    CASE WHEN py.avg_price_eur = 0 THEN NULL ELSE (cy.avg_price_eur / py.avg_price_eur) - 1 END AS yoy_avg_price_pct,
    CASE WHEN py.commission_rate = 0 THEN NULL ELSE (cy.commission_rate / py.commission_rate) - 1 END AS yoy_commission_rate_pct,
    CASE WHEN py.conversion_rate = 0 THEN NULL ELSE (cy.conversion_rate / py.conversion_rate) - 1 END AS yoy_conversion_rate_pct,
    CASE WHEN py.new_visitor_share = 0 THEN NULL ELSE (cy.new_visitor_share / py.new_visitor_share) - 1 END AS yoy_new_visitor_share_pct,
    CASE WHEN py.bookings_with_coupon_share = 0 THEN NULL ELSE (cy.bookings_with_coupon_share / py.bookings_with_coupon_share) - 1 END AS yoy_bookings_with_coupon_share_pct

  FROM cy
  LEFT JOIN py
    ON py.quarter_start = add_months(cy.quarter_start, -12)
),

/* -----------------------------
   Long format with formatting
   IMPORTANT:
   - conversion_rate, take_rate_*, shares are FRACTIONS -> multiply by 100 for %
   - commission_rate is already in PERCENT POINTS (10 = 10%) -> DO NOT multiply by 100
   - yoy_commission_rate_abs is already pp -> DO NOT multiply by 100
   ----------------------------- */
long AS (
  SELECT
    quarter_label,
    metric,
    value
  FROM final
  LATERAL VIEW stack(
    90,

    -- CY (non-%)
    'cy_sessions',                   cast(round(sessions, 0) as double),
    'cy_visitors',                   cast(round(visitors, 0) as double),
    'cy_new_visitors',               cast(round(new_visitors, 0) as double),
    'cy_returning_visitors',         cast(round(returning_visitors, 0) as double),

    'cy_bookings',                   cast(round(bookings, 0) as double),
    'cy_transactions',               cast(round(transactions, 0) as double),
    'cy_bookings_with_coupon',       cast(round(bookings_with_coupon, 0) as double),

    'cy_gmv_eur',                    cast(round(gmv_eur, 0) as double),
    'cy_nr_eur',                     cast(round(nr_eur, 0) as double),
    'cy_nr_bt_eur',                  cast(round(nr_bt_eur, 0) as double),
    'cy_net_tax_eur',                cast(round(net_tax_eur, 0) as double),

    'cy_aov_eur',                    cast(round(aov_eur, 2) as double),
    'cy_avg_price_eur',              cast(round(avg_price_eur, 2) as double),

    -- CY (%)
    'cy_new_visitor_share',          cast(round(new_visitor_share * 100, 2) as double),
    'cy_bounce_rate',                cast(round(bounce_rate * 100, 2) as double),
    'cy_quoter_rate',                cast(round(quoter_rate * 100, 2) as double),
    'cy_take_rate_nr',               cast(round(take_rate_nr * 100, 2) as double),
    'cy_take_rate_nr_bt',            cast(round(take_rate_nr_bt * 100, 2) as double),

    -- CY commission rate (already %)
    'cy_commission_rate',            cast(round(commission_rate, 2) as double),

    'cy_conversion_rate',            cast(round(conversion_rate * 100, 2) as double),
    'cy_bookings_with_coupon_share', cast(round(bookings_with_coupon_share * 100, 2) as double),

    -- PY (non-%)
    'py_sessions',                   cast(round(py_sessions, 0) as double),
    'py_visitors',                   cast(round(py_visitors, 0) as double),
    'py_new_visitors',               cast(round(py_new_visitors, 0) as double),
    'py_returning_visitors',         cast(round(py_returning_visitors, 0) as double),

    'py_bookings',                   cast(round(py_bookings, 0) as double),
    'py_transactions',               cast(round(py_transactions, 0) as double),
    'py_bookings_with_coupon',       cast(round(py_bookings_with_coupon, 0) as double),

    'py_gmv_eur',                    cast(round(py_gmv_eur, 0) as double),
    'py_nr_eur',                     cast(round(py_nr_eur, 0) as double),
    'py_nr_bt_eur',                  cast(round(py_nr_bt_eur, 0) as double),
    'py_net_tax_eur',                cast(round(py_net_tax_eur, 0) as double),

    'py_aov_eur',                    cast(round(py_aov_eur, 2) as double),
    'py_avg_price_eur',              cast(round(py_avg_price_eur, 2) as double),

    -- PY (%)
    'py_new_visitor_share',          cast(round(py_new_visitor_share * 100, 2) as double),
    'py_bounce_rate',                cast(round(py_bounce_rate * 100, 2) as double),
    'py_quoter_rate',                cast(round(py_quoter_rate * 100, 2) as double),
    'py_take_rate_nr',               cast(round(py_take_rate_nr * 100, 2) as double),
    'py_take_rate_nr_bt',            cast(round(py_take_rate_nr_bt * 100, 2) as double),

    -- PY commission rate (already %)
    'py_commission_rate',            cast(round(py_commission_rate, 2) as double),

    'py_conversion_rate',            cast(round(py_conversion_rate * 100, 2) as double),
    'py_bookings_with_coupon_share', cast(round(py_bookings_with_coupon_share * 100, 2) as double),

    -- YoY abs (non-%)
    'yoy_sessions_abs',              cast(round(yoy_sessions_abs, 0) as double),
    'yoy_visitors_abs',              cast(round(yoy_visitors_abs, 0) as double),
    'yoy_new_visitors_abs',          cast(round(yoy_new_visitors_abs, 0) as double),
    'yoy_returning_visitors_abs',    cast(round(yoy_returning_visitors_abs, 0) as double),

    'yoy_bookings_abs',              cast(round(yoy_bookings_abs, 0) as double),
    'yoy_transactions_abs',          cast(round(yoy_transactions_abs, 0) as double),
    'yoy_bookings_with_coupon_abs',  cast(round(yoy_bookings_with_coupon_abs, 0) as double),

    'yoy_gmv_eur_abs',               cast(round(yoy_gmv_eur_abs, 0) as double),
    'yoy_nr_eur_abs',                cast(round(yoy_nr_eur_abs, 0) as double),
    'yoy_nr_bt_eur_abs',             cast(round(yoy_nr_bt_eur_abs, 0) as double),
    'yoy_net_tax_eur_abs',           cast(round(yoy_net_tax_eur_abs, 0) as double),

    'yoy_aov_eur_abs',               cast(round(yoy_aov_eur_abs, 2) as double),
    'yoy_avg_price_eur_abs',         cast(round(yoy_avg_price_eur_abs, 2) as double),

    -- abs deltas on rates -> percentage points
    'yoy_new_visitor_share_abs_pp',          cast(round(yoy_new_visitor_share_abs * 100, 2) as double),

    -- commission pp delta (already pp)
    'yoy_commission_rate_abs_pp',            cast(round(yoy_commission_rate_abs, 2) as double),

    'yoy_conversion_rate_abs_pp',            cast(round(yoy_conversion_rate_abs * 100, 2) as double),
    'yoy_bookings_with_coupon_share_abs_pp', cast(round(yoy_bookings_with_coupon_share_abs * 100, 2) as double),

    -- YoY % (%)
    'yoy_sessions_pct',              cast(round(yoy_sessions_pct * 100, 2) as double),
    'yoy_visitors_pct',              cast(round(yoy_visitors_pct * 100, 2) as double),
    'yoy_new_visitors_pct',          cast(round(yoy_new_visitors_pct * 100, 2) as double),
    'yoy_returning_visitors_pct',    cast(round(yoy_returning_visitors_pct * 100, 2) as double),

    'yoy_bookings_pct',              cast(round(yoy_bookings_pct * 100, 2) as double),
    'yoy_transactions_pct',          cast(round(yoy_transactions_pct * 100, 2) as double),
    'yoy_bookings_with_coupon_pct',  cast(round(yoy_bookings_with_coupon_pct * 100, 2) as double),

    'yoy_gmv_eur_pct',               cast(round(yoy_gmv_eur_pct * 100, 2) as double),
    'yoy_nr_eur_pct',                cast(round(yoy_nr_eur_pct * 100, 2) as double),
    'yoy_nr_bt_eur_pct',             cast(round(yoy_nr_bt_eur_pct * 100, 2) as double),
    'yoy_net_tax_eur_pct',           cast(round(yoy_net_tax_eur_pct * 100, 2) as double),

    'yoy_aov_pct',                   cast(round(yoy_aov_pct * 100, 2) as double),
    'yoy_avg_price_pct',             cast(round(yoy_avg_price_pct * 100, 2) as double),
    'yoy_commission_rate_pct',       cast(round(yoy_commission_rate_pct * 100, 2) as double),
    'yoy_conversion_rate_pct',       cast(round(yoy_conversion_rate_pct * 100, 2) as double),
    'yoy_new_visitor_share_pct',     cast(round(yoy_new_visitor_share_pct * 100, 2) as double),
    'yoy_bookings_with_coupon_share_pct', cast(round(yoy_bookings_with_coupon_share_pct * 100, 2) as double)

  ) s AS metric, value
)

SELECT *
FROM long
PIVOT (
  max(value) FOR quarter_label IN (
    'Q1 - 2026'
  )
)
ORDER BY metric;
